# 🎬 MiniMax H3 (Hailuo) em Google Colab Free (Tesla T4 15 GB)
### Pipeline de Prova de Conceito (PoC) Low-VRAM

**Configuração do Ambiente:**
- **GPU:** Tesla T4 (15 GB VRAM) + ~12.7 GB RAM do Sistema
- **Engine:** ComfyUI com `--lowvram` + CPU Offloading
- **Modelo:** MiniMax H3 GGUF (Quantizado Q3_K_M / Q4_K_M)
- **Target:** Resolução reduzida (480p / 512x512), ~3–4 segundos (~49 frames), testando viabilidade rápida.

--- 
## 📦 Passo 1 — Instalação do ComfyUI e Gerenciadores de Rede

In [ ]:
# 1. Clonar ComfyUI
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt -q
!pip install -U huggingface_hub aria2 -q

# 2. Instalar nós essenciais para GGUF e H3
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!git clone https://github.com/chflame163/ComfyUI-MiniMaxH3-Easy.git

# 3. Baixar Cloudflared para criar o túnel de acesso Web público
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

%cd /content/ComfyUI
print("✅ ComfyUI e Custom Nodes instalados com sucesso!")

--- 
## 🔍 Passo 2 — Verificar Recursos (GPU e Espaço em Disco)

In [ ]:
!df -h /content
!nvidia-smi

--- 
## 📥 Passo 3 — Download Estratégico dos Modelos H3 GGUF (Low-Footprint)

> **Estratégia para a T4 (15 GB VRAM):**
> - **DiT (Diffusion Transformer):** `Q3_K_M` (~9.8 GB) ou `Q4_K_M` (~13 GB) em `/content/ComfyUI/models/unet`
> - **Text Encoder (Qwen3-VL):** `Q2_K` / `Q4_K_M` (~6–10 GB) em `/content/ComfyUI/models/clip`
> - **VAE Oficial:** `vae` em `/content/ComfyUI/models/vae`

In [ ]:
import os
from huggingface_hub import hf_hub_download

# Diretórios de destino
UNET_DIR = "/content/ComfyUI/models/unet"
CLIP_DIR = "/content/ComfyUI/models/clip"
VAE_DIR  = "/content/ComfyUI/models/vae"
os.makedirs(UNET_DIR, exist_ok=True)
os.makedirs(CLIP_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)

print("📥 Baixando MiniMax H3 GGUF quantizado (Otimizado para T4 15GB)...")

# 1. DiT Model (Q3_K_M ou Q4_K_M)
!aria2c -c -x 16 -s 16 -d /content/ComfyUI/models/unet \
  https://huggingface.co/unsloth/MiniMax-H3-GGUF/resolve/main/MiniMax-H3-fl2va-Q3_K_M.gguf

# 2. Text Encoder & Projector (Qwen3-VL)
!aria2c -c -x 16 -s 16 -d /content/ComfyUI/models/clip \
  https://huggingface.co/unsloth/MiniMax-H3-GGUF/resolve/main/qwen3_vl_32b_q2_k.gguf

# 3. VAE
!aria2c -c -x 16 -s 16 -d /content/ComfyUI/models/vae \
  https://huggingface.co/MiniMax-AI/MiniMax-H3/resolve/main/vae/diffusion_pytorch_model.safetensors -o minimax_h3_vae.safetensors

print("✅ Todos os modelos foram baixados com sucesso!")

--- 
## 🚀 Passo 4 — Iniciar o ComfyUI em modo Low-VRAM com Túnel Público

In [ ]:
import subprocess
import threading
import time
import re

# 1. Iniciar túnel Cloudflare em segundo plano
def run_cloudflared():
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in iter(p.stdout.readline, ''):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n" + "="*60)
            print(f"🔗 ACESSE O COMFYUI AQUI: {match.group(0)}")
            print("="*60 + "\n")

threading.Thread(target=run_cloudflared, daemon=True).start()
time.sleep(2)

# 2. Iniciar ComfyUI com flags otimizadas para 15 GB VRAM
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188 --lowvram --preview-method auto --disable-cuda-malloc

--- 
## ⚙️ Configurações Recomendadas no Workflow para a Prova de Conceito (T4)

| Parâmetro | Valor Recomendado para Teste PoC | Motivo |
| :--- | :--- | :--- |
| **Resolução** | `512x512` ou `480x270` | Minimiza o footprint de ativação da GPU no DiT |
| **Frames** | `33` a `49` frames (~2 a 3 segundos @ 16fps) | Evita estouro de VRAM na fase de decodificação VAE |
| **Steps** | `15` a `20` steps | Reduz o tempo de iteração inicial |
| **Sampler / Scheduler** | `Euler` / `Simple` ou `Normal` | Mais leve e estável em quantizações GGUF |
| **CFG** | `4.0` a `6.0` | Boa fidelidade ao prompt sem artefatos excessivos |